In [1]:
import os
import torch
import librosa
import numpy as np
import pretty_midi
import torch.nn as nn
import torch.nn.functional as F 

In [7]:
class PianoTranscriptionNet(nn.Module):
    def __init__(self, input_bins=88, lstm_units=256):
        super(PianoTranscriptionNet, self).__init__()
        
        # 1. Shared CNN Acoustic Layer
        # We use padding to ensure the temporal dimension (Width) stays exactly the same size.
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding=(1, 1))
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=(1, 1))
        self.bn2 = nn.BatchNorm2d(64)
        
        # After flattening the channel and frequency dimensions, calculate the feature size per frame.
        # Channels (64) * Frequency Bins (88) = 5632
        cnn_out_features = 64 * input_bins
        
        # 2. Onset Detection Head
        self.onset_lstm = nn.LSTM(cnn_out_features, lstm_units, batch_first=True, bidirectional=True)
        self.onset_fc = nn.Linear(lstm_units * 2, input_bins) # * 2 because it's bidirectional
        
        # 3. Frame Detection Head
        # The Frame head takes the raw CNN features AND the predicted onset states as input.
        # Input size: cnn_out_features (5632) + predicted onsets (88) = 5720
        self.frame_lstm = nn.LSTM(cnn_out_features + input_bins, lstm_units, batch_first=True, bidirectional=True)
        self.frame_fc = nn.Linear(lstm_units * 2, input_bins)

    def forward(self, x):
        # x shape: [Batch, 1, 88, TimeFrames]
        batch_size, channels, bins, time_frames = x.size()
        
        # Run through CNN
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out))) # Shape: [Batch, 64, 88, TimeFrames]
        
        # Reshape for LSTM sequence processing
        # Target shape: [Batch, TimeFrames, Channels * Bins]
        out = out.permute(0, 3, 1, 2).contiguous()
        out = out.view(batch_size, time_frames, -1)
        
        # --- Head 1: Predict Onsets ---
        onset_out, _ = self.onset_lstm(out)
        onset_logits = self.onset_fc(onset_out) # Shape: [Batch, TimeFrames, 88]
        onset_probs = torch.sigmoid(onset_logits)
        
        # --- Head 2: Predict Frames (Conditioned on Onset Probabilities) ---
        # Concatenate the acoustic features with the predicted onset values along the feature axis
        frame_input = torch.cat([out, onset_probs], dim=-1)
        
        frame_out, _ = self.frame_lstm(frame_input)
        frame_logits = self.frame_fc(frame_out) # Shape: [Batch, TimeFrames, 88]
        frame_probs = torch.sigmoid(frame_logits)
        
        # Rearrange outputs back to match dataset targets: [Batch, 88, TimeFrames]
        return frame_probs.permute(0, 2, 1), onset_probs.permute(0, 2, 1)

In [10]:
def transcribe_audio(audio_path, model_path, output_midi_path):
    # --- 1. Environmental Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Constants matching your training pipeline exactly
    TARGET_SR = 16000
    HOP_LENGTH = 256
    N_BINS = 88
    FMIN = librosa.note_to_hz('C1')
    FRAME_TIME = HOP_LENGTH / TARGET_SR # ~0.016s per frame
    
    # --- 2. Load and Preprocess Audio ---
    print("Loading and extracting features from audio...")
    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    C = librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH, fmin=FMIN, n_bins=N_BINS)
    C_db = librosa.amplitude_to_db(np.abs(C), ref=np.max)
    
    # Format tensor: [Batch=1, Channels=1, Bins=88, Frames]
    input_tensor = torch.tensor(C_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    # --- 3. Run Inference ---
    print("Running neural network inference...")
    # Initialize your model architecture structure
    model = PianoTranscriptionNet(input_bins=N_BINS, lstm_units=256)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        pred_frames, pred_onsets = model(input_tensor)
        
    # Remove batch dimension and move back to CPU numpy arrays
    pred_frames = pred_frames.squeeze(0).cpu().numpy() # Shape: [88, TotalFrames]
    pred_onsets = pred_onsets.squeeze(0).cpu().numpy() # Shape: [88, TotalFrames]
    
    total_frames = pred_frames.shape[1]
    
    # --- 4. Decode Probabilities into MIDI Events ---
    print("Decoding states into MIDI notes...")
    pm = pretty_midi.PrettyMIDI()
    piano_program = pretty_midi.instrument_name_to_program('Acoustic Grand Piano')
    piano = pretty_midi.Instrument(program=piano_program)
    
    # Thresholding Hyperparameters (Tune these to change sensitivity!)
    ONSET_THRESHOLD = 0.35
    FRAME_THRESHOLD = 0.20
    
    # Track the active status of all 88 keys
    # Stores the start frame if a note is currently active, or None if inactive
    active_notes = {key_idx: None for key_idx in range(88)}
    
    for frame_idx in range(total_frames):
        current_time = frame_idx * FRAME_TIME
        
        for key_idx in range(88):
            midi_note_number = key_idx + 21 # Translate back to absolute MIDI range
            
            onset_prob = pred_onsets[key_idx, frame_idx]
            frame_prob = pred_frames[key_idx, frame_idx]
            
            # Case A: Note is currently inactive, look for an Onset trigger
            if active_notes[key_idx] is None:
                if onset_prob > ONSET_THRESHOLD:
                    active_notes[key_idx] = current_time
                    
            # Case B: Note is currently active, check if it should be released
            else:
                if frame_prob < FRAME_THRESHOLD:
                    start_time = active_notes[key_idx]
                    end_time = current_time
                    
                    # Prevent zero-duration "ghost notes"
                    if end_time - start_time > 0.03: 
                        note_event = pretty_midi.Note(
                            velocity=100, # Static velocity baseline
                            pitch=midi_note_number,
                            start=start_time,
                            end=end_time
                        )
                        piano.notes.append(note_event)
                        
                    # Reset key status back to inactive
                    active_notes[key_idx] = None
                    
    # Append instrument data to file stream and write to disk
    pm.instruments.append(piano)
    pm.write(output_midi_path)
    print(f"🎉 Transcription successfully saved to: {output_midi_path}")


In [11]:
transcribe_audio(
    audio_path="inputs/La La Land.mp3", 
    model_path="best_piano_transcriber.pth", 
    output_midi_path="outputs/transcribed_output.mid"
)

Loading and extracting features from audio...
Running neural network inference...
Decoding states into MIDI notes...
🎉 Transcription successfully saved to: outputs/transcribed_output.mid
